#  Praktikum 3: GPU Programming in Triton

The aim of the lab is understanding GPU programming. After completing this lab,
* you will have a mental model of how parallel programs run on GPU.
* How to implement, test, benchmark, and tune GPU implementations.

by implementing following tasks.
1. Write optimized matrix multiplication on GPU in Triton.
2. Test, benchmark, and tune for throughput on a modern GPU hardware.

## Setup

Triton is a python-based domain specific language for writing GPU code.

### Prerequisites

 The following code installs triton. The notebook requires a CUDA-capable GPU. Open the notebook on a CUDA box (Colab, a workstation with an NVIDIA GPU, EFI GPU Cluster or a cloud instance). It is *not* runnable on macOS or CPU-only machines. The DEVICE should show CUDA if GPU is selected as an accelerator.

In [13]:
try:
    import torch
    assert torch.cuda.is_available()
except (ImportError, AssertionError):
    !pip install -q torch --index-url https://download.pytorch.org/whl/cu124
    import torch

!pip install -q triton torchprofile

import triton
import triton.language as tl
import time

torch.manual_seed(0); torch.cuda.manual_seed_all(0)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


### Utilities

Following simple helper functions can be used for benchmarking your gpu implementation of matrix multiplication and measuring the throughput in GFLOPS (Assumption: Square Matrix).

In [14]:
# Usage Example: bench(lambda: simple_matmul(A, B))
def bench(fn, warmup=2, reps=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = time.time()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    end = time.time()
    return (end - start) * 1000 / reps  # ms

def gflops(N, time_ms):
    """GFLOPS for an N x N square matmul: FLOPs = 2 * N^3."""
    return 2 * N**3 / (time_ms * 1e-3) / 1e9

def kernel_info(jit_fn):
    """Return compile-time stats for any cached variant of jit_fn.
    Walks whichever cache structure this Triton version uses."""
    # Triton has used: `cache`, `device_caches` — find whichever exists.
    cache = None
    for attr in ("device_caches", "cache"):
        if hasattr(jit_fn, attr):
            cache = getattr(jit_fn, attr)
            break
    if not cache:
        return None

    # The cache is a nested mix of dicts / tuples / lists ending in a
    # CompiledKernel (which has a `.metadata` attribute). Walk recursively.
    def find_compiled(obj, depth=0):
        if depth > 5 or obj is None:
            return None
        if hasattr(obj, "metadata") and not isinstance(obj, (dict, list, tuple)):
            return obj
        if isinstance(obj, dict):
            for v in obj.values():
                r = find_compiled(v, depth + 1)
                if r is not None:
                    return r
        elif isinstance(obj, (tuple, list)):
            for v in obj:
                r = find_compiled(v, depth + 1)
                if r is not None:
                    return r
        return None

    k = find_compiled(cache)
    if k is None:
        return None
    md = k.metadata
    return dict(
        n_regs       = getattr(k,  "n_regs",     None),
        n_spills     = getattr(k,  "n_spills",   None),
        shared_bytes = getattr(md, "shared",     None),
        num_warps    = getattr(md, "num_warps",  None),
        num_stages   = getattr(md, "num_stages", None),
    )

## GPU Characterzation

**Task 1**

Find out the following information about the GPU from torch (`torch.cuda.get_device_properties(0)`), datasheet, or wikipedia.
* Peak Performance (FP32): Compute peak FP32 from first principles using your card's SM count, FP32 cuda cores per SM, and clock.
* Peak Performance (FP16)
* Peak DRAM bandwidth (GB/s)
* Shared Memory per Streaming Multiprocessor

In [15]:
props = torch.cuda.get_device_properties(0)
GPU_NAME = props.name

# TODO (Task 1): Fill in the three values below.
#   PEAK_FP32_TFLOPS — derive from first principles:
#       num_SMs * fp32_cores_per_SM * 2 (FMA = 2 FLOPs) * boost_clock_GHz
#   PEAK_FP16_TFLOPS — tensor-core peak from datasheet
#   PEAK_BW          — peak DRAM bandwidth from datasheet (GB/s)

# GPU | SMs(DS2) | FP32 cores/SM (DS3) | Boost(GHz) | FP32 TFLOPS(DS1) | FP16 Tensor TFLOPS | BW (GB/s) | SMEM/SM (KB)
# T4  | 40       | 64                  | 1.59       | 8.1              | 65                 | 300       | 64
# datasheet1 (DS1): https://www.nvidia.com/content/dam/en-zz/Solutions/Data-Center/tesla-t4/t4-tensor-core-datasheet-951643.pdf
# datasheet2 (DS2): https://images.nvidia.com/aem-dam/en-zz/Solutions/design-visualization/technologies/turing-architecture/NVIDIA-Turing-Architecture-Whitepaper.pdf
# datasheet3 (DS3): https://docs.nvidia.com/cuda/turing-tuning-guide/index.html


num_SMs = 40
fp32_cores_per_SM = 64
boost_clock_GHz = 1.59

PEAK_FP32_TFLOPS_DS1 = 8.1
PEAK_FP32_TFLOPS = num_SMs * fp32_cores_per_SM * 2 * boost_clock_GHz / 1000
PEAK_FP16_TFLOPS = 65     # Mixed-Precision (FP16/FP32) from datasheet1
PEAK_BW          = 300    # DRAM bandwidth (GB/s) from datasheet1

print(f"GPU                         : {GPU_NAME}")
print(f"Compute Capability          : {props.major}.{props.minor}")
print(f"Streaming Multiprocessors   : {props.multi_processor_count}")
print(f"VRAM (total)                : {props.total_memory / 1024**3:.2f} GiB")
print(f"L2 Cache                    : {props.L2_cache_size / 1024**2:.2f} MiB")

print(f"\nPeak FP32 (datasheet)       : {PEAK_FP32_TFLOPS_DS1} TFLOPS")
print(f"Peak FP32 (calculated)      : {PEAK_FP32_TFLOPS} TFLOPS")
print(f"Peak FP16 (tensor cores)    : {PEAK_FP16_TFLOPS} TFLOPS")
print(f"Peak DRAM BW (datasheet)    : {PEAK_BW} GB/s")


GPU                         : Tesla T4
Compute Capability          : 7.5
Streaming Multiprocessors   : 40
VRAM (total)                : 14.56 GiB
L2 Cache                    : 4.00 MiB

Peak FP32 (datasheet)       : 8.1 TFLOPS
Peak FP32 (calculated)      : 8.1408 TFLOPS
Peak FP16 (tensor cores)    : 65 TFLOPS
Peak DRAM BW (datasheet)    : 300 GB/s


## Matrix Multiplication on GPU

A simple matrix multiplication $C[m,n] = \sum A[m,k] * B[k,n]$ loads each element of A and B once per output element. Following is a Triton kernel that launches a 2-D grid of size `(N, N)`.
Assumption:
* Square matrix
* fp32 datatype

In [16]:
@triton.jit
def simple_matmul_kernel(a_ptr, b_ptr, c_ptr, N):
    pid_m = tl.program_id(0)   # row index
    pid_n = tl.program_id(1)   # column index

    acc = 0.0

    for k in range(N):
        a_val = tl.load(a_ptr + pid_m * N + k)
        b_val = tl.load(b_ptr + k * N + pid_n)
        acc += a_val * b_val

    tl.store(c_ptr + pid_m * N + pid_n, acc)

def simple_matmul(a, b):
    N = a.shape[0]
    c = torch.empty((N, N), device=a.device, dtype=a.dtype)
    simple_matmul_kernel[(N, N)](a, b, c, N)

    return c

**Task 2**
* Run and verify correctness against `torch.matmul` for `N = 64`.
  - Correctness check passed for N = 64
* Execution Model: Which part of the above code runs on CPU and GPU respectively?
  - simple_matmul (allocating output tensor, launching the kernel, passing pointers) on CPU
  - simple_matmul_kernel on GPU (actual computation)
  - Bus: transferring pointers/arguments from CPU to GPU (the data A, B, C already lives on GPU since device=DEVICE)
* SIMT Execution Model: How many triton programs are launched for `N = 64`?
  - The grid is (N, N) = (64, 64) = 4,096 programs. Each program computes exactly one element of C.
* SIMT Execution Model: What is the program_id of triton program which calculates C[1,1]:     
    - pid_m = tl.program_id(0) = 1   # row index
    - pid_n = tl.program_id(1) = 1   # column index
* Memory Model: What is total number of load and store from one triton program.
  - Each program loops k = 0..63, doing 2 loads per iteration (one from A, one from B), plus 1 store at the end:
    - Loads: 64 × 2 = 128
    - Stores: 1
    - Total: 129 memory operations per program
* Memory Model: What is the total number of floating point operation per triton program
  - Each loop iteration does 1 multiply + 1 add (the acc += a_val * b_val is a fused multiply-add). Over 64 iterations:
  - 128 FLOPs per program (64 multiplies + 64 adds)
* Arithmetic intensity (AI): how many total bytes are moved? What is the (FLOPs / byte) of this kernel?
  - Total across all 4,096 programs:
    - Total FLOPs: 4,096 × 128 = 524,288 FLOPs
    - Total loads: 4,096 × 128 = 524,288 loads, each loading a float32 (4 bytes) = 2,097,152 bytes loaded
    - Total stores: 4,096 × 1 = 4,096 stores × 4 bytes = 16,384 bytes stored
    - Total bytes moved: 2,097,152 + 16,384 = 2,113,536 bytes ≈ 2.01 MiB
    - Arithmetic Intensity: 524,288 / 2,113,536 ≈ 0.248 FLOPs/byte
      -> Memory-bound! Reason: Crossover point between memory and compute bound is for FP32 in this case: 8.1 TFLOPS / 300 GB/s ≈ 27 FLOPs/byte. Values below that point are memory-bound.

In [17]:
N = 64
A = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5

torch.testing.assert_close(simple_matmul(A, B), torch.matmul(A, B), atol=1e-2, rtol=0)
print("Correctness check passed for N =", N)

Correctness check passed for N = 64


# Benchmarking Performance

**Task 3**
* Measure mean GPU kernel time for naive matrix multiplication (N=1024) in milliseconds.
  - 466.1 ms. That's very slow for a GPU matrix multiply
* What is the throughput of the naive GPU implementation in gflops?
  - 4.6 GFLOPS
* Is it higher compared to Optimized CPU implementation in Lab 2? Compare to your card's peak performance derived in Task 1.
  - Optimized CPU 53.553.5 Gflops / 154.4 Gflops/s = 34.6% (Values calculated in Assignment 2)
  - Naive GPU (T4) 4.64.6 Gflops/ 8,100 Gflops/s = 0.057% (8100 Gflops from datasheet 1)
  - Naive GPU kernel achieves only 0.057% of the T4's FP32 peak. It's not only far from the GPU's potential, it's actually ~12x slower than the optimized CPU implementation.
* kernel_info return compile-time stats for any triton matmul kernel, showing register and shared memory usage. What is the reason that the Naive implementation is slow.
  - The kernel_info explains the problem: shared_bytes: 0. The kernel uses no shared memory, meaning every value is loaded from global VRAM with no reuse. Each of the 1,024 × 1,024 = 1,048,576 programs loads an entire row of A and column of B (1,024 loads each) independently. The same row of A gets loaded 1,024 times by different programs, and the same column of B gets loaded 1,024 times. That's a massive amount of redundant global memory traffic.

In [18]:
N = 1024
A = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
info = kernel_info(simple_matmul_kernel)

print(f"Registers used per Thread            : {info['n_regs']}")
print(f"Register spills                      : {info['n_spills']}")
print(f"Shared memory per SM                 : {info['shared_bytes'] / 1024} KB")
print(f"Number of warps per triton programm  : {info['num_warps']}")
print(f"Number of threads per triton programm: {info['num_warps'] * 32}")
print(f"Software pipeline stages             : {info['num_stages']}")

t = bench(lambda: simple_matmul(A, B))
# TODO:
print(f"Kernel execution time: {t} ms")
print(f"Gflops: {gflops(N, t)} ")

Registers used per Thread            : 21
Register spills                      : 0
Shared memory per SM                 : 0.0 KB
Number of warps per triton programm  : 4
Number of threads per triton programm: 128
Software pipeline stages             : 3
Kernel execution time: 548.9915490150452 ms
Gflops: 3.9116879883721993 


# Optimization: Vectorization

The kernel `naive_matmul_kernel` upgrades the scalar loop in `simple_matmul_kernel` to a **vectorized** loop. Instead of loading one float per iteration and adding seuquentially. It loads a vector of `BLOCK_K` floats, multiplies element-wise, and reduces with `tl.sum`.

**Task 4**
* Run, verify, and benchmark correctness against `torch.matmul` for `N = 64`. By what factor does vectorization speed things up?
  - Works, Kernel Execution time is now 0.073ms and 7.1 Gflops -> Speedup of 5.947,9x (with using 64 for BLOCK_K, going below that reduces speedup)
* SIMT Execution Model: How many triton programs are launched for `N = 64`?
  - Still 4096: Each program still computes one element of C, but now the k-loop is vectorized, loading and operating on BLOCK_K=64 elements at once, which keeps 64 of 128 threads busy instead of just 1."
* SIMT Execution Model: How many threads are actually doing useful work per program? How is the dot product computed over time (step-by-step work)?
  - 64 threads load a row-slice of A and a column-slice of B from global memory. Then 64 of 128 threads perform element-wise multiply (acc += a * b), producing 64 partial products. A tree reduction (tl.sum) sums these in 6 steps (32 → 16 → 8 → 4 → 2 → 1 active threads) to produce a single scalar. Finally, 1 thread stores the result to C.
  - This is done 4096 times until the whole result matrix of C is calculated and stored
* What is the fundamental bottleneck that vectorization alone cannot fix?
  - Each row of A gets loaded 64 times (once for every program in that row of C), and each column of B gets loaded 64 times too. There's no data reuse between programs. Every program independently fetches its own data from global memory and throws it away. Instead the first row of could be used 64 times for computing the whole first row of C.

* Interesting Accident:
  - Tested a Triton matmul kernel with N = 64 while accidentally keeping BLOCK_K = 128.
  - What happened: The kernel still executed without crashing (I used 1024 for N before)
  - Performance metrics (GFLOPS, runtime, compiler stats) stayed similar and bytes_shared was on 16 instead of 8 for BLOCK_K = 64 and N = 64
  - However after a few retries, correctness broke severely (~97% mismatches vs PyTorch)

In [19]:
@triton.jit
def naive_matmul_kernel(a_ptr, b_ptr, c_ptr, N, BLOCK_K: tl.constexpr):
    # TODO (Task 4): vectorized naive matmul kernel.
    # Each program still computes ONE output element C[pid_m, pid_n], but the
    # K-loop should now process BLOCK_K elements at a time using `tl.arange`,
    # then reduce with `tl.sum`. This keeps all the threads in the program busy
    # instead of having 127 of 128 threads idle (as in simple_matmul_kernel).
    #
    # Suggested skeleton:
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    acc = tl.zeros((BLOCK_K,), dtype=tl.float32)
    for k in range(0, N, BLOCK_K):
        offs_k = k + tl.arange(0, BLOCK_K)
        a = tl.load(a_ptr + pid_m * N + offs_k)       # row pid_m of A,  K-slice -> shape (BLOCK_K,)
        b = tl.load(b_ptr + offs_k * N + pid_n)       # col pid_n of B,  K-slice -> shape (BLOCK_K,)
        acc += a * b

    result = tl.sum(acc, axis=0)
    tl.store(c_ptr + pid_m * N + pid_n, result)
    pass


def naive_matmul(a, b, BLOCK_K=64):
    N = a.shape[0]
    c = torch.empty((N, N), device=a.device, dtype=torch.float32)
    grid = (N, N)
    naive_matmul_kernel[grid](a, b, c, N, BLOCK_K=BLOCK_K, num_warps=4)
    return c


# --- Verify and benchmark ---
N = 64
A = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5

# Uncomment after implementing the kernel above:
torch.testing.assert_close(naive_matmul(A, B), torch.matmul(A, B), atol=1e-2, rtol=0)
print("naive_matmul correctness OK")
info = kernel_info(naive_matmul_kernel)
print(info)
t = bench(lambda: naive_matmul(A, B))
print(f"Kernel execution time: {t:.3f} ms")
print(f"GFLOPS: {gflops(N, t):.1f}")


naive_matmul correctness OK
{'n_regs': 27, 'n_spills': 0, 'shared_bytes': 8, 'num_warps': 4, 'num_stages': 3}
Kernel execution time: 0.050 ms
GFLOPS: 10.5


# Optimization: Tiled Matrix Multiplication

The naive kernel 100× below peak performance. It spends almost all its time waiting for memory, not computing. Instead of computing one output element per program, assign each program a **`BLOCK × BLOCK`** output tile of C. The computation $C = A \cdot B$ can be broken into panels:

**Task 5**
* Rewrite the kernel to compute a `BLOCK × BLOCK` output tile per program, using `tl.dot` for block-level reuse.
* Verify correctness. - Verrified
* What is the arithmetic intensity in BLOCK = 64?
  - Arithmetic intensity (AI): how many total bytes are moved? What is the (FLOPs / byte) of this kernel?
    - Total across all 1 program (just one programm for N=64):
    - Total FLOPs: acc += tl.dot(a, b) -> 2 × BLOCK^3 (×N/BLOCK) = 2 × N^3 (N=BLOCK here) = 2 × 64^3 = 524,288 FLOPs
    - Total loads: (N/BLOCK) × 2 × BLOCK² × 4 = 1 (iterations in k-loop) × 2 = 2 loads, each loading 4096 float32 (4 bytes) => 2 × 64^2 × 4 = 32.768 bytes loaded
    - Total stores: 1 × BLOCK^2 × 4 = 1 store × 64^2 × 4 bytes/float32 = 16,384 bytes stored
    - Total bytes moved: 32,768 + 16,384 = 49,152 bytes
    - Arithmetic Intensity: 524,288 / 49,152 ≈ 10.67 FLOPs/byte -> Still Memory-bound for N=64 and Block=64! Reason: Crossover point between memory and compute bound is for FP32 in this case: 8.1 TFLOPS / 300 GB/s ≈ 27 FLOPs/byte. Values below that point are memory-bound.
* Assumptions: Square Matrices, Square tiles

In [20]:
@triton.jit
def simple_tiled_matmul_kernel(a_ptr, b_ptr, c_ptr, N, BLOCK: tl.constexpr):
    # TODO (Task 5): tiled matmul kernel.
    # Each program computes a BLOCK x BLOCK output tile of C using `tl.dot`
    # so the inner GEMM runs at block granularity (and on tensor cores when
    # available). The K-loop walks K in BLOCK-sized chunks.
    #
    # Suggested skeleton:
    pid_m = tl.program_id(0)         # which row-tile of C
    pid_n = tl.program_id(1)         # which col-tile of C

    offs_m = pid_m * BLOCK + tl.arange(0, BLOCK)
    offs_n = pid_n * BLOCK + tl.arange(0, BLOCK)

    acc = tl.zeros((BLOCK, BLOCK), dtype=tl.float32)
    for k in range(0, N, BLOCK):
        offs_k = k + tl.arange(0, BLOCK)
        a = tl.load(a_ptr + offs_m[:, None] * N + offs_k[None, :])   # (BLOCK, BLOCK)
        b = tl.load(b_ptr + offs_k[:, None] * N + offs_n[None, :])   # (BLOCK, BLOCK)
        acc += tl.dot(a, b)

    tl.store(c_ptr + offs_m[:, None] * N + offs_n[None, :], acc)
    pass


def simple_tiled_matmul(a, b, BLOCK=64):
    N = a.shape[0]
    c = torch.empty((N, N), device=a.device, dtype=torch.float32)
    grid = (N // BLOCK, N // BLOCK)
    simple_tiled_matmul_kernel[grid](
        a, b, c,
        N,
        BLOCK=BLOCK,
        num_warps=4,
    )
    return c


In [21]:
# Testing Tiled Matrix Multiplication
A = torch.rand((64, 64), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((64, 64), device=DEVICE, dtype=torch.float32) - 0.5

torch.testing.assert_close(
    simple_tiled_matmul(A, B, 64),
    torch.matmul(A, B),
    atol=1e-1, rtol=1e-2,  # ← also loosen tolerance
)
print("OK")
t = bench(lambda: simple_tiled_matmul(A, B))
print(f"Kernel execution time: {t:.3f} ms")
print(f"GFLOPS: {gflops(N, t):.1f}")

OK
Kernel execution time: 0.065 ms
GFLOPS: 8.1


**Task 6** Benchmarking Tiled Matrix Multiplication

1. Try out different Block size in multiple of 2?What block size leads to failure?
  - 8: Input shapes should have M >= 1, N >= 1 and K >= 16 - GPU (Turing architecture), the hardware requires the K dimension to be at least 16
  - 16: works - Kernel execution time: 6.466948986053467 ms
Gflops: 2656.564822306449
  - 32: works - Kernel execution time: 6.330265998840332 ms
Gflops: 2713.9253211709038
  - 64: works - Kernel execution time: 5.936229228973389 ms
Gflops: 2894.0710544244066
  - 128: out of resource: shared memory, Required: 131072, Hardware limit: 65536. Reducing block sizes or `num_stages` may help.
  - 256: out of resource: shared memory, Required: 524288, Hardware limit: 65536. Reducing block sizes or `num_stages` may help.
2. Assuming FP32, what is the SRAM Usage per SM for the above block size
- per iteration it loads...
  - ... 16^2×4×2 = 2048 Bytes for BLOCK = 16
  - ... 32^2×4×2 = 8192 Bytes for BLOCK = 32
  - ... 32768 Bytes for BLOCK 64
  - ... 131072 Bytes for BLOCK 128
  - ... 524288 Bytes for BLOCK 256
3. What block size (in powers of 2) and N = 2048 leads to highest throughput?
- 64
4. Is this kernel compute bound or memory bound?
- With the formulas for Arithmetic Intensity:
  	- Total across all 1 program (Here 1024 Programms)
    - Total FLOPs: acc += tl.dot(a, b) -> 2 × BLOCK^3 (×N/BLOCK) = 2 × 64^3 × 32 = 16,777,216 FLOPs
    - Total loads: (N/BLOCK) × 2 × BLOCK² × 4 = 32 (iterations in k-loop) × 2 × 64^2 × 4 = 1,048,576 bytes loaded
    - Total stores: 1 × BLOCK^2 × 4 = 1 store × 64^2 × 4 bytes/float32 = 16.384 bytes stored
    - Total bytes moved: 1,048,576 + 16,384 = 1.064.960 bytes
    - Arithmetic Intensity: 16,777,216 / 1.064.960 ≈ 15.75 FLOPs/byte -> Still Memory-bound for N=64 and Block=64! Reason: Crossover point between memory and compute bound is for FP32 in this case: 8.1 TFLOPS / 300 GB/s ≈ 27 FLOPs/byte. Values below that point are memory-bound.

In [30]:
N = 2048
A = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5

# Run kernel
t = bench(lambda: simple_tiled_matmul(A, B, 64), 50, 200)
t = bench(lambda: torch.matmul(A,B), 50,200)

print(f"Kernel execution time: {t} ms")
print(f"Gflops: {gflops(N, t)} ")

Kernel execution time: 6.23967170715332 ms
Gflops: 2753.3290195868085 


In [ ]:

A = torch.rand((2048, 2048), device="cuda", dtype=torch.float32)
B = torch.rand((2048, 2048), device="cuda", dtype=torch.float32)
simple_tiled_matmul(A,B,  64)
torch.cuda.synchronize()

info = kernel_info(simple_tiled_matmul_kernel)
print(info)

# Benchmarking

**Task 7**
1. Use the `@triton.testing.perf_report` / `Benchmark` decorator (see Triton tutorials) to sweep `N ∈ [256, 4096]` and plot TFLOPS for `simple_tiled_matmul` vs `torch.matmul` (cuBLAS).
2. Explain how tiled matrix multiplication is running on GPU, based on compile time information based on kernel_info for `BLOCK = 32` and `BLOCK = 64`

  - In tiled matmul, each program computes a BLOCK × BLOCK output tile of C. Per k-loop iteration, it loads a tile from A and B into shared memory, multiplies them with tl.dot, and accumulates the result. Each loaded element is reused BLOCK times, giving AI ≈ BLOCK/4. So BLOCK=32 yields 8 FLOPs/byte, BLOCK=64 yields 16 FLOPs/byte (both still memory-bound, threshold is 27). BLOCK=64 also launches 4× fewer programs, reducing overhead and roughly doubling throughput. BLOCK=128 exceeds the T4's 64 KB shared memory limit, making BLOCK=64 optimal. Performance drops at large N are due to thermal throttling.
  - Comparing it to kernel_info:
    - BLOCK = 32: 
      - Registers used per Thread            : 128
      - Register spills                      : 0
      - Shared memory per SM                 : 32.0 KB
      - Number of warps per triton programm  : 4
      - Number of threads per triton programm: 128
      - Software pipeline stages             : 3
    - BLOCK = 64:
      - Registers used per Thread            : 128
      - Register spills                      : 0
      - Shared memory per SM                 : 32.0 KB
      - Number of warps per triton programm  : 4
      - Number of threads per triton programm: 128
      - Software pipeline stages             : 3
- Note: Kaggle, Colab GPUs have thermal throttling by reducing clock speed.

In [ ]:
import triton
import triton.testing

configs = [
    triton.testing.Benchmark(
        x_names      = ["N"],
        x_vals       = [256 * i for i in range(2, 16)],   # 256, 384, ..., 4096
        line_arg     = "provider",
        line_vals    = ["torch", "tiled"],
        line_names   = ["cuBLAS (torch.matmul)", "Triton tiled (BLOCK=64)"],
        styles       = [("green", "-"), ("blue", "-")],
        ylabel       = "TFLOPS",
        plot_name    = "matmul-fp32-square",
        args         = {},
    ),
]


@triton.testing.perf_report(configs)
def benchmark(N, provider):
    a = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
    b = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5

    quantiles = [0.5, 0.2, 0.8]

    if provider == "torch":
        fn = lambda: torch.matmul(a, b)
    elif provider == "tiled":
        fn = lambda: simple_tiled_matmul(a,b, BLOCK=64)
        bench(lambda: simple_tiled_matmul(a, b, 64), 50, 200) 
        info = kernel_info(simple_tiled_matmul_kernel)
    ms, min_ms, max_ms = triton.testing.do_bench(fn, quantiles=quantiles)
    perf = lambda ms: 2 * N * N * N * 1e-12 / (ms * 1e-3)   # TFLOPS
    return perf(ms), perf(max_ms), perf(min_ms)


benchmark.run(show_plots=True, print_data=True)
print(f"Registers used per Thread            : {info['n_regs']}")
print(f"Register spills                      : {info['n_spills']}")
print(f"Shared memory per SM                 : {info['shared_bytes'] / 1024} KB")
print(f"Number of warps per triton programm  : {info['num_warps']}")
print(f"Number of threads per triton programm: {info['num_warps'] * 32}")
print(f"Software pipeline stages             : {info['num_stages']}")

In [ ]:
torch.cuda.empty_cache()
N = 2048
A = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5
B = torch.rand((N, N), device=DEVICE, dtype=torch.float32) - 0.5

# Single shot, fresh thermal state
ms = triton.testing.do_bench(lambda: simple_tiled_matmul(A, B, BLOCK=64),
                              warmup=10, rep=200, quantiles=None)
!nvidia-smi --query-gpu=name,power.limit,power.max_limit,clocks.max.gr \
            --format=csv,noheader
print(f"{N=}, {ms:.2f} ms, {2*N**3/(ms*1e-3)/1e12:.2f} TFLOPS")

# Profiling and Analysis
Following code uses `torch.profiler.profile` to capture a few iterations of the tiled kernel.

**Task 8** Analysis
1. For a 10-iteration loop of your kernel at `N = 4096`, what fraction of total time is GPU compute vs CPU overhead?
2. Set TRACE_PATH and visualize the traces with perfetto (https://ui.perfetto.dev/)
2. Using GPU's `regs_per_multiprocessor`, `shared_memory_per_multiprocessor`, and `max_threads_per_multi_processor`, compute theoretical occupancy for `BLOCK = 64`
3. Below which `N` does your kernel under-use the SMs?
4. For `BLOCK = 64` and `num_warps = 4`, How many output elements does one thread compute?


In [ ]:
import torch.profiler, pathlib

N_PROF   = 2048
A_prof   = torch.rand((N_PROF, N_PROF), device=DEVICE, dtype=torch.float32)
B_prof   = torch.rand((N_PROF, N_PROF), device=DEVICE, dtype=torch.float32)
TRACE_PATH = "/kaggle/working/triton_matmul_trace.json" #or ./triton_matmul_trace.json

# warmup
for _ in range(5):
    simple_tiled_matmul(A_prof, B_prof, BLOCK=64)
torch.cuda.synchronize()

with torch.profiler.profile(
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    record_shapes=True,
) as prof:
    for _ in range(10):
        with torch.profiler.record_function("triton_tiled_BLOCK64"):
            simple_tiled_matmul(A_prof, B_prof, BLOCK=64)
    torch.cuda.synchronize()


print("\nTop 5 ops/kernels by cuda_time_total:")
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=5))


prof.export_chrome_trace(TRACE_PATH)
print(f"\nChrome trace written to: {TRACE_PATH}")

# Ideas for Future Work
- Swizzling for better L2 Cache Usage
- Roofline model and placement
- does an FP32-input matmul use the tensor cores? What is TF32
- Auto-tuning.
- Add an FP16-input variant of your tiled kernel
- profile with nsys and ncu
- Integrate in Mobilenet_v1